# P4-3 · Bài 6 · Đánh giá hệ thống AI

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

## Lý thuyết

## Bài tập 4: Bộ chạy eval {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import json, statistics, time

def run_eval(path, search, answer, judge, k=5, out="eval_results.jsonl"):
    rows = [json.loads(l) for l in open(path, encoding="utf-8")]
    results = []
    for r in rows:
        t0 = time.time()
        hits = search(r["question"], k)
        ans, _ = answer(r["question"])
        verdict = judge(r["question"], ans, r["reference"])           # {"reason", "score"}
        results.append({**r, "recall": len(set(hits) & set(r["gold_ids"])) / max(1, len(r["gold_ids"])),
                        "answer": ans, **verdict, "latency": time.time() - t0})
    with open(out, "w", encoding="utf-8") as f:
        for x in results:
            f.write(json.dumps(x, ensure_ascii=False) + "\n")
    print(f"recall@{k} {statistics.mean(x['recall'] for x in results):.3f} | "
          f"điểm TB {statistics.mean(x['score'] for x in results):.2f} | "
          f"tỉ lệ điểm 3 {sum(x['score'] == 3 for x in results) / len(results):.0%}")
    return results

## Bài tập 5: Đo retrieval trên bộ eval vàng {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import pandas as pd
gold = [("Học bổng loại giỏi bao nhiêu?", "học bổng", {"hb1"}), ("Khi nào công bố học bổng?", "học bổng", {"hb3"}),
        ("Điểm rèn luyện tính thế nào?", "rèn luyện", {"rl1", "rl2"}), ("Nghỉ học quá bao nhiêu buổi thì cấm thi?", "thi", {"t2"}),
        ("Phúc khảo bài thi ở đâu?", "thi", {"t5"}), ("Đăng ký học lại thế nào?", "đăng ký", {"dk2"}),
        ("Học phí học lại tính ra sao?", "đăng ký", {"dk4"}), ("Thi lại mấy lần?", "thi", {"t3"})]
he_thong = [["hb1", "hb2", "rl1", "t1", "dk1"], ["hb2", "hb1", "dk3", "hb3", "t1"], ["rl2", "rl3", "rl1", "hb1", "t2"],
            ["t1", "t4", "t3", "dk1", "t2"], ["t1", "t2", "t3", "t4", "dk1"], ["dk2", "dk1", "dk3", "t1", "hb1"],
            ["dk1", "dk3", "dk2", "hb2", "t3"], ["t4", "t1", "t2", "t5", "t6"]]
# TODO: bảng df với cột nhom, recall3, recall5, rr; tổng hợp theo nhóm

In [ ]:
rows = []
for (cau, nhom, dung), kq in zip(gold, he_thong):
    rr = next((1 / (i + 1) for i, d in enumerate(kq) if d in dung), 0.0)
    rows.append({"nhom": nhom, "recall3": len(dung & set(kq[:3])) / len(dung),
                 "recall5": len(dung & set(kq[:5])) / len(dung), "rr": rr})
df = pd.DataFrame(rows)
print(df.mean(numeric_only=True).round(3))
print(df.groupby("nhom")[["recall3", "recall5", "rr"]].mean().round(3))

recall3    0.375
recall5    0.625
rr         0.431
dtype: float64
           recall3  recall5     rr
nhom                              
học bổng       0.5    1.000  0.625
rèn luyện      1.0    1.000  1.000
thi            0.0    0.333  0.067
đăng ký        0.5    0.500  0.500


## Bài tập 6: Chấm câu trả lời bằng token F1 {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import re
from collections import Counter
cap = [("Học bổng loại giỏi bằng 120% học phí.", "120% học phí", 1),
       ("Mức học bổng giỏi là 120 phần trăm học phí của kỳ đó", "120% học phí", 1),
       ("Sinh viên được thi lại 2 lần.", "Được thi lại tối đa 2 lần", 1),
       ("Phúc khảo tại phòng đào tạo trong 5 ngày.", "Nộp đơn tại phòng khảo thí trong 7 ngày", 0),
       ("Không có thông tin.", "Nghỉ quá 20% số buổi thì bị cấm thi", 0)]

def chuan_hoa(s):
    return re.sub(r"[^\w\s%]", "", s.lower()).split()

def f1(pred, ref):
    # TODO: token F1 giữa hai câu
    return 0.0

In [ ]:
def f1(pred, ref):
    p, r = chuan_hoa(pred), chuan_hoa(ref)
    chung = sum((Counter(p) & Counter(r)).values())
    if chung == 0:
        return 0.0
    P, R = chung / len(p), chung / len(r)
    return 2 * P * R / (P + R)

diem = [f1(p, r) for p, r, _ in cap]
dong_y = sum((d >= 0.5) == bool(nguoi) for d, (_, _, nguoi) in zip(diem, cap)) / len(cap)
for (p, r, nguoi), d in zip(cap, diem):
    print(f"F1 {d:.2f} | người chấm {nguoi} | {p}")
print("đồng ý với người chấm:", dong_y)

F1 0.55 | người chấm 1 | Học bổng loại giỏi bằng 120% học phí.
F1 0.25 | người chấm 1 | Mức học bổng giỏi là 120 phần trăm học phí của kỳ đó
F1 0.71 | người chấm 1 | Sinh viên được thi lại 2 lần.
F1 0.56 | người chấm 0 | Phúc khảo tại phòng đào tạo trong 5 ngày.
F1 0.00 | người chấm 0 | Không có thông tin.
đồng ý với người chấm: 0.6


## Bài tập 7: Bắt thiên vị vị trí của LLM-as-judge {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
chat_luong = rng.normal(0, 1, (200, 2))          # chất lượng thật của câu A và câu B trong mỗi cặp

def fake_judge(q_dau, q_sau):
    """Trả về 'dau' hoặc 'sau'. Thiên vị +0,8 cho câu đứng đầu."""
    return "dau" if q_dau - q_sau + 0.8 + rng.normal(0, 0.5) > 0 else "sau"
# TODO: lan1 (A đứng đầu), lan2 (B đứng đầu); mau_thuan; ti_le_A_thang (chỉ tính cặp nhất quán)

In [ ]:
lan1 = [fake_judge(a, b) for a, b in chat_luong]
lan2 = [fake_judge(b, a) for a, b in chat_luong]
A_thang = [(x == "dau") and (y == "sau") for x, y in zip(lan1, lan2)]
B_thang = [(x == "sau") and (y == "dau") for x, y in zip(lan1, lan2)]
mau_thuan = 1 - (sum(A_thang) + sum(B_thang)) / len(lan1)
ti_le_A_thang = sum(A_thang) / max(1, sum(A_thang) + sum(B_thang))
print(f"lần 1 chọn câu đầu: {np.mean([x == 'dau' for x in lan1]):.0%}")
print(f"tự mâu thuẫn: {mau_thuan:.0%}; trong các cặp nhất quán, A thắng {ti_le_A_thang:.0%}")

lần 1 chọn câu đầu: 68%
tự mâu thuẫn: 38%; trong các cặp nhất quán, A thắng 48%


## Bài tập 8: Bản mới có thật sự tốt hơn: bootstrap {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np, pandas as pd
rng = np.random.default_rng(1)
n = 120
nhom = rng.choice(["chính sách", "sản phẩm", "giao hàng"], n, p=[0.4, 0.35, 0.25])
v1 = (rng.random(n) < 0.70).astype(int)
v2 = v1.copy()
v2[np.flatnonzero((v1 == 0) & (nhom != "giao hàng"))[:12]] = 1    # v2 sửa đúng được 12 câu v1 sai
v2[np.flatnonzero((v1 == 1) & (nhom == "giao hàng"))[:6]] = 0     # nhưng làm sai 6 câu giao hàng v1 đã đúng
# TODO: chenh (v2 - v1 trung bình), ci (2.5% và 97.5% của 5000 lần bootstrap), bảng lỗi v2 theo nhóm

In [ ]:
chenh = v2.mean() - v1.mean()
rng_b = np.random.default_rng(0)
ds = []
for _ in range(5000):
    idx = rng_b.integers(0, n, n)
    ds.append(v2[idx].mean() - v1[idx].mean())
ci = np.percentile(ds, [2.5, 97.5])
print(f"v1 {v1.mean():.3f}, v2 {v2.mean():.3f}, chênh {chenh:+.3f}, khoảng tin cậy 95%: [{ci[0]:+.3f}, {ci[1]:+.3f}]")
print(pd.DataFrame({"nhom": nhom, "v1_sai": 1 - v1, "v2_sai": 1 - v2}).groupby("nhom").mean().round(2))

v1 0.692, v2 0.742, chênh +0.050, khoảng tin cậy 95%: [-0.017, +0.117]
            v1_sai  v2_sai
nhom                      
chính sách    0.31    0.18
giao hàng     0.35    0.53
sản phẩm      0.27    0.12
